# Huấn luyện model với cơ chế Early Stopping
Mục tiêu:
- Khôi phục lại tập dữ liệu Train/Validation chống rò rỉ (từ step 1).
- Khôi phục lại mô hình VGG16 đã tinh chỉnh (từ step 3).
- Thiết lập thuật toán tối ưu (Optimizer) **chỉ tập trung học ở phần classifier**.
- Chạy vòng lặp huấn luyện, tự động theo dõi `Validation Loss` để lưu lại trọng số tốt nhất và ngắt sớm (Early Stopping) nếu mô hình có dấu hiệu học vẹt (Overfitting).

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm
import os
import copy
import time

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Sử dụng thiết bị CUDA: {device}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Sử dụng thiết bị Apple Metal Performance Shaders: {device}")
else:
    device = torch.device("cpu")
    print(f"Sử dụng CPU: {device}")

Sử dụng thiết bị Apple Metal Performance Shaders: mps


## Nạp lại Dữ liệu (Dùng file phân chia từ step 1)
Ta phải khai báo lại các Transform chuẩn của VGG16, tải dataset và dùng `Subset` cùng file `indices.pt` để trích xuất chính xác 100% những bức ảnh đã chia ở Notebook 1, đảm bảo không một bức ảnh Test nào bị lọt vào đây.

In [2]:
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Transform (Khai báo lại giống hệt step 1)
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Tải Data gốc
raw_train = datasets.FashionMNIST(root='./data', train=True, download=True, transform=train_transform)
raw_val = datasets.FashionMNIST(root='./data', train=True, download=True, transform=eval_transform)

# Đọc file chia dữ liệu (từ thư mục data_splits của NB1)
train_idx = torch.load('data_splits/train_indices.pt', weights_only=False)
val_idx = torch.load('data_splits/val_indices.pt', weights_only=False)

train_loader = DataLoader(Subset(raw_train, train_idx), batch_size=32, shuffle=True)
val_loader = DataLoader(Subset(raw_val, val_idx), batch_size=32, shuffle=False)

print(f"Đã nạp {len(train_idx)} ảnh Train và {len(val_idx)} ảnh Validation.")

Đã nạp 48000 ảnh Train và 12000 ảnh Validation.


## Nạp lại Mô hình và Thiết lập Optimizer
Ta khởi tạo lại VGG16, đổi lớp cuối thành 10 class, và load file trọng số đã lưu ở Notebook 3.

Thuật toán tối ưu (Optimizer) chỉ được phép nhận vào `model.classifier.parameters()` để tránh việc nó cố gắng thay đổi phần thân đã bị đóng băng.

In [3]:
# Khởi tạo mô hình
model = models.vgg16()
model.classifier[6] = nn.Linear(model.classifier[6].in_features, 10)

# Tải trọng số từ step 3
model.load_state_dict(torch.load("models/vgg16_untrained_custom_head.pth", weights_only=True))
model = model.to(device)

# Hàm tính sai số (Cross Entropy Loss cho phân loại nhiều lớp)
criterion = nn.CrossEntropyLoss()

# Thuật toán tối ưu: Adam. Chỉ truyền tham số của phần classifier
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

print("Đã nạp mô hình và thiết lập thuật toán tối ưu.")

Đã nạp mô hình và thiết lập thuật toán tối ưu.


## Vòng lặp Huấn luyện (Kết hợp Early Stopping)
Cơ chế bảo vệ mô hình hoạt động như sau:
- Sau mỗi vòng (Epoch), kiểm tra điểm số trên tập Validation.
- Nếu điểm Val_Loss giảm (tốt hơn) -> Lưu lại cấu hình mạng ngay lập tức.
- Nếu điểm Val_Loss tăng (tồi đi) trong `patience` vòng liên tiếp (ví dụ 5 vòng) -> Lập tức ngắt huấn luyện, vì AI đang bắt đầu "học vẹt".

In [4]:
num_epochs = 20
patience = 5
best_val_loss = float('inf')
early_stop_counter = 0

# Khởi tạo bản sao lưu trọng số tốt nhất
best_model_weights = copy.deepcopy(model.state_dict())

print("Tiến trình huấn luyện:")

for epoch in range(num_epochs):
    start_time = time.time()

    # Training phase
    model.train()
    running_loss, running_corrects = 0.0, 0

    # Bọc tqdm để hiển thị tiến trình chạy trực tiếp và thời gian còn lại (ETA)
    for inputs, labels in tqdm(train_loader, desc=f"Train Epoch {epoch+1:02}/{num_epochs}", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    train_loss = running_loss / len(train_idx)
    train_acc = running_corrects.float() / len(train_idx)

    # Validation phase
    model.eval()
    val_loss, val_corrects = 0.0, 0

    with torch.no_grad():
        # Bọc tqdm cho pha Validation
        for inputs, labels in tqdm(val_loader, desc=f"Val Epoch {epoch+1:02}/{num_epochs}", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            _, preds = torch.max(outputs, 1)
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)

    val_loss = val_loss / len(val_idx)
    val_acc = val_corrects.float() / len(val_idx)

    epoch_time = time.time() - start_time

    # In dòng thông số tổng kết của Epoch
    print(f"Epoch {epoch+1:02}/{num_epochs} [{epoch_time:.0f}s] | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    # Kiểm tra điều kiện dừng sớm (Early Stopping)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(model.state_dict())
        early_stop_counter = 0
        print("  --> Kiểm tra: Val_loss cải thiện. Đã cập nhật trọng số tốt nhất!")
    else:
        early_stop_counter += 1
        print(f"  --> Kiểm tra: Val_loss không cải thiện ({early_stop_counter}/{patience})")

    if early_stop_counter >= patience:
        print(f"\nEarly-stopped! Dừng huấn luyện ở Epoch {epoch+1} để ngăn ngừa overfitting.")
        break

# Khôi phục lại cấu hình tối ưu nhất trước khi kết thúc
model.load_state_dict(best_model_weights)

# Lưu mô hình hoàn chỉnh xuống thư mục models/
os.makedirs("models", exist_ok=True)
final_save_path = "models/vgg16_best_transfer_learning.pth"
torch.save(model.state_dict(), final_save_path)

print(f"\nFile mô hình tốt nhất đã nằm tại: {final_save_path}")

Tiến trình huấn luyện:


Epoch 01/20 [2158s] | Train Loss: 0.6991 Acc: 0.8044 | Val Loss: 0.3473 Acc: 0.8888
  --> Kiểm tra: Val_loss cải thiện. Đã cập nhật trọng số tốt nhất!


Epoch 02/20 [2002s] | Train Loss: 0.5576 Acc: 0.8391 | Val Loss: 0.3330 Acc: 0.8985
  --> Kiểm tra: Val_loss cải thiện. Đã cập nhật trọng số tốt nhất!


Epoch 03/20 [2019s] | Train Loss: 0.5035 Acc: 0.8533 | Val Loss: 0.3128 Acc: 0.9072
  --> Kiểm tra: Val_loss cải thiện. Đã cập nhật trọng số tốt nhất!


Epoch 04/20 [1948s] | Train Loss: 0.4765 Acc: 0.8584 | Val Loss: 0.2934 Acc: 0.9056
  --> Kiểm tra: Val_loss cải thiện. Đã cập nhật trọng số tốt nhất!


...

Epoch 14/20 [1880s] | Train Loss: 0.3850 Acc: 0.8849 | Val Loss: 0.2797 Acc: 0.9213
  --> Kiểm tra: Val_loss không cải thiện (1/5)


Epoch 15/20 [1882s] | Train Loss: 0.3771 Acc: 0.8887 | Val Loss: 0.2944 Acc: 0.9207
  --> Kiểm tra: Val_loss không cải thiện (2/5)


Epoch 16/20 [1906s] | Train Loss: 0.3560 Acc: 0.8923 | Val Loss: 0.2948 Acc: 0.9193
  --> Kiểm tra: Val_loss không cải thiện (3/5)


Epoch 17/20 [1947s] | Train Loss: 0.3707 Acc: 0.8921 | Val Loss: 0.2785 Acc: 0.9170
  --> Kiểm tra: Val_loss không cải thiện (4/5)


Epoch 18/20 [2003s] | Train Loss: 0.3664 Acc: 0.8892 | Val Loss: 0.2839 Acc: 0.9164
  --> Kiểm tra: Val_loss không cải thiện (5/5)

Early-stopped! Dừng huấn luyện ở Epoch 18 để ngăn ngừa overfitting.

File mô hình tốt nhất đã nằm tại: models/vgg16_best_transfer_learning.pth
